In [31]:
import os
import cv2
import pytesseract
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import time
from datetime import datetime
import traceback

print('✓ All libraries imported!')

✓ All libraries imported!


In [32]:
import pytesseract

# Try to get version
try:
    # First, set the path (update if different)
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    
    version = pytesseract.get_tesseract_version()
    print(f'✓ SUCCESS! Tesseract is installed')
    print(f'✓ Version: {version}')
except Exception as e:
    print(f'✗ FAILED: {e}')
    print(f'\n⚠️ Tesseract is not installed or path is wrong')
    print(f'\nTry installing from: https://github.com/UB-Mannheim/tesseract/wiki')

✓ SUCCESS! Tesseract is installed
✓ Version: 5.5.3.20260724


In [33]:
print('\n' + '='*70)
print('🔍 TESSERACT VERIFICATION')
print('='*70)

paths = [
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
   
]

print('\nSearching for Tesseract...')
found_path = None
for path in paths:
    exists = os.path.exists(path)
    status = '✓' if exists else '✗'
    print(f'  {status} {path}')
    if exists:
        found_path = path
        break

if found_path:
    print(f'\n✓ FOUND: {found_path}')
    pytesseract.pytesseract.pytesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    
    # Verify
    try:
        version = pytesseract.get_tesseract_version()
        print(f'✓ Version: {version}')
        TESSERACT_OK = True
    except Exception as e:
        print(f'⚠ Version check failed: {e}')
        TESSERACT_OK = False
else:
    print(f'\n❌ TESSERACT NOT FOUND!')
    print('\n📥 Installation required:')
    print('   1. Download: https://github.com/UB-Mannheim/tesseract/wiki')
    print('   2. Run installer and select default path')
    print('   3. Or update path above if installed elsewhere')
    TESSERACT_OK = False

print('\n' + '='*70)


🔍 TESSERACT VERIFICATION

Searching for Tesseract...
  ✓ C:\Program Files\Tesseract-OCR\tesseract.exe

✓ FOUND: C:\Program Files\Tesseract-OCR\tesseract.exe
✓ Version: 5.5.3.20260724



In [34]:
# Settings
Base_directory = r'C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR'
train_dir = os.path.join(Base_directory, 'training')
test_dir = os.path.join(Base_directory, 'testing')

print('\n' + '='*70)
print('⚙️  CONFIGURATION')
print('='*70)
print(f'\nBase: {Base_directory}')
print(f'Train: {train_dir}')
print(f'  Exists: {os.path.exists(train_dir)}')
if os.path.exists(train_dir):
    count = len([f for f in os.listdir(train_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    print(f'  Images: {count}')

print(f'\nTest: {test_dir}')
print(f'  Exists: {os.path.exists(test_dir)}')
if os.path.exists(test_dir):
    count = len([f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    print(f'  Images: {count}')

print('\n' + '='*70)


⚙️  CONFIGURATION

Base: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR
Train: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\training
  Exists: True
  Images: 4

Test: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing
  Exists: True
  Images: 4



In [35]:
# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import easyocr
import pytesseract
import cv2
from PIL import Image
import numpy as np
from pathlib import Path
import time
import json
from datetime import datetime

print("✓ All libraries imported successfully!")

# ============================================================
# STEP 2: CONFIGURE DATA DIRECTORIES
# ============================================================

# 🔴 REPLACE THESE WITH YOUR ACTUAL PATHS

Base_directory = r'C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR'
TRAINING_DATA_DIR = os.path.join(Base_directory, 'training')
TESTING_DATA_DIR = os.path.join(Base_directory, 'testing')
OUTPUT_DIR = os.path.join(Base_directory, 'ocr_results')
# TRAINING_DATA_DIR = './training_data'      # Where you store training images
# TESTING_DATA_DIR = './testing_data'        # Where you store test images
# OUTPUT_DIR = './ocr_results'               # Where results are saved

# Create directories if they don't exist
Path(TRAINING_DATA_DIR).mkdir(exist_ok=True)
Path(TESTING_DATA_DIR).mkdir(exist_ok=True)
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"✓ Directories configured:")
print(f"  Training data: {TRAINING_DATA_DIR}")
print(f"  Testing data: {TESTING_DATA_DIR}")
print(f"  Results: {OUTPUT_DIR}")

# ============================================================
# STEP 3: DEFINE TEST IMAGES (TESTING DATA)
# ============================================================

# 🔴 REPLACE THESE WITH YOUR IMAGE FILENAMES
TEST_IMAGES = [
    r'C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6208.jpg',
    r"C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6209.jpg",
    r"C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6212.jpg",
    r"C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6213.jpg"
]

# Or auto-find all images:
# TEST_IMAGES = list(Path(TESTING_DATA_DIR).glob('*.jpg')) + \
#               list(Path(TESTING_DATA_DIR).glob('*.png'))

print(f"\n✓ Test images configured: {len(TEST_IMAGES)} images")
for img in TEST_IMAGES:
    print(f"   - {img}")

# ============================================================
# STEP 4: INITIALIZE OCR ENGINES
# ============================================================

print("\n" + "="*70)
print("INITIALIZING OCR ENGINES")
print("="*70)

# EasyOCR with Arabic + English
print("🔄 Loading EasyOCR (Arabic + English)...")
easyocr_reader = easyocr.Reader(['ar', 'en'], gpu=False)
print("✓ EasyOCR Ready")

# Tesseract is already installed on system
print("✓ Tesseract Ready")

# ============================================================
# STEP 5: IMAGE PREPROCESSING
# ============================================================

def preprocess_image(image_path):
    """
    Preprocess image for better OCR results
    Optimized for Arabic + English text
    """
    image = cv2.imread(str(image_path))
    if image is None:
        return None, None
    
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Enhance contrast (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    # Denoise
    denoised = cv2.fastNlMeansDenoising(enhanced, h=10)
    
    # Binary threshold
    _, binary = cv2.threshold(denoised, 150, 255, cv2.THRESH_BINARY)
    
    return binary, cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ============================================================
# STEP 6: EASYOCR EXTRACTION (ARABIC + ENGLISH)
# ============================================================

def extract_with_easyocr(image_rgb):
    """
    Extract text using EasyOCR
    Automatically handles Arabic + English
    """
    results = easyocr_reader.readtext(image_rgb)
    
    # Extract text and confidence
    text_data = []
    for detection in results:
        text = detection[1]
        confidence = detection[2]
        text_data.append({
            'text': text,
            'confidence': confidence
        })
    
    full_text = "\n".join([item['text'] for item in text_data])
    avg_confidence = np.mean([item['confidence'] for item in text_data])
    
    return full_text, avg_confidence, text_data

# ============================================================
# STEP 7: TESSERACT EXTRACTION (ARABIC + ENGLISH)
# ============================================================

def extract_with_tesseract(processed_image):
    """
    Extract text using Tesseract OCR
    Configured for Arabic + English bilingual text
    """
    try:
        # 🔴 KEY CONFIG FOR ARABIC + ENGLISH:
        # lang='ara+eng' = Both languages
        # --psm 6 = Assume single uniform block of text
        
        text = pytesseract.image_to_string(
            processed_image,
            lang='ara+eng',          # ← LANGUAGE CONFIG
            config='--psm 6'
        )
        
        return text, True
    except Exception as e:
        print(f"❌ Tesseract error: {e}")
        return "", False

# ============================================================
# STEP 8: LANGUAGE ANALYSIS
# ============================================================

def analyze_language_composition(text):
    """
    Analyze Arabic vs English content in extracted text
    """
    total_chars = len(text)
    if total_chars == 0:
        return {
            'arabic_chars': 0,
            'english_chars': 0,
            'arabic_percent': 0,
            'english_percent': 0,
            'total_words': 0
        }
    
    # Arabic Unicode range: U+0600 to U+06FF
    arabic_chars = len([c for c in text if '\u0600' <= c <= '\u06FF'])
    english_chars = len([c for c in text if c.isascii() and c.isalpha()])
    
    return {
        'total_chars': total_chars,
        'total_words': len(text.split()),
        'arabic_chars': arabic_chars,
        'english_chars': english_chars,
        'arabic_percent': round((arabic_chars/total_chars)*100, 1) if total_chars > 0 else 0,
        'english_percent': round((english_chars/total_chars)*100, 1) if total_chars > 0 else 0
    }

# ============================================================
# STEP 9: PROCESS SINGLE IMAGE
# ============================================================

def process_single_image(image_path, verbose=True):
    """
    Process one image with both OCR methods
    """
    
    if verbose:
        print(f"\n📍 Processing: {Path(image_path).name}")
        print("-"*70)
    
    # Check if file exists
    if not Path(image_path).exists():
        print(f"❌ File not found: {image_path}")
        return None
    
    # Preprocess
    processed, rgb_image = preprocess_image(image_path)
    if processed is None:
        print(f"❌ Could not load image")
        return None
    
    # ========================
    # METHOD 1: EasyOCR
    # ========================
    if verbose:
        print("🔄 EasyOCR (Arabic + English)...")
    
    start_time = time.time()
    easyocr_text, easyocr_conf, easyocr_details = extract_with_easyocr(rgb_image)
    easyocr_time = time.time() - start_time
    easyocr_stats = analyze_language_composition(easyocr_text)
    
    if verbose:
        print(f"  ✓ Complete in {easyocr_time:.2f}s")
        print(f"  ✓ {easyocr_stats['total_words']} words extracted")
        print(f"  ✓ Arabic: {easyocr_stats['arabic_percent']}% | English: {easyocr_stats['english_percent']}%")
        print(f"  ✓ Confidence: {easyocr_conf:.2%}")
    
    # ========================
    # METHOD 2: Tesseract
    # ========================
    if verbose:
        print("\n🔄 Tesseract (Arabic + English)...")
    
    start_time = time.time()
    tesseract_text, success = extract_with_tesseract(processed)
    tesseract_time = time.time() - start_time
    tesseract_stats = analyze_language_composition(tesseract_text) if success else None
    
    if success and verbose:
        print(f"  ✓ Complete in {tesseract_time:.2f}s")
        print(f"  ✓ {tesseract_stats['total_words']} words extracted")
        print(f"  ✓ Arabic: {tesseract_stats['arabic_percent']}% | English: {tesseract_stats['english_percent']}%")
    
    # ========================
    # RESULTS
    # ========================
    results = {
        'image_name': Path(image_path).name,
        'timestamp': datetime.now().isoformat(),
        'easyocr': {
            'text': easyocr_text,
            'stats': easyocr_stats,
            'confidence': float(easyocr_conf),
            'time': easyocr_time
        },
        'tesseract': {
            'text': tesseract_text,
            'stats': tesseract_stats,
            'time': tesseract_time,
            'success': success
        }
    }
    
    return results

# ============================================================
# STEP 10: PROCESS ALL TEST IMAGES (BATCH)
# ============================================================

def process_all_test_images():
    """
    Process all test images at once
    """
    print("\n" + "="*70)
    print("🚀 BATCH PROCESSING TEST IMAGES")
    print("="*70)
    
    all_results = []
    
    for idx, image_file in enumerate(TEST_IMAGES, 1):
        print(f"\n[{idx}/{len(TEST_IMAGES)}]", end=" ")
        
        result = process_single_image(image_file, verbose=True)
        if result:
            all_results.append(result)
    
    return all_results

# ============================================================
# STEP 11: SAVE RESULTS
# ============================================================

def save_results(results, output_dir='ocr_results'):
    """
    Save extracted text and metadata for each image
    """
    print("\n" + "="*70)
    print("💾 SAVING RESULTS")
    print("="*70)
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    for result in results:
        image_name = Path(result['image_name']).stem
        
        # Save EasyOCR text
        easyocr_file = output_dir / f"{image_name}_EasyOCR.txt"
        with open(easyocr_file, 'w', encoding='utf-8') as f:
            f.write(result['easyocr']['text'])
        print(f"✓ {easyocr_file.name}")
        
        # Save Tesseract text
        if result['tesseract']['success']:
            tesseract_file = output_dir / f"{image_name}_Tesseract.txt"
            with open(tesseract_file, 'w', encoding='utf-8') as f:
                f.write(result['tesseract']['text'])
            print(f"✓ {tesseract_file.name}")
        
        # Save metadata (JSON)
        metadata_file = output_dir / f"{image_name}_metadata.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"✓ {metadata_file.name}")
    
    # Save comprehensive report
    save_comparison_report(results, output_dir)

# ============================================================
# STEP 12: GENERATE COMPARISON REPORT
# ============================================================

def save_comparison_report(results, output_dir='ocr_results'):
    """
    Create detailed comparison report of all images
    """
    output_dir = Path(output_dir)
    report_file = output_dir / "COMPARISON_REPORT.txt"
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("OCR COMPARISON REPORT: EasyOCR vs Tesseract\n")
        f.write("Languages: Arabic + English\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")
        
        for idx, result in enumerate(results, 1):
            f.write(f"\n[IMAGE {idx}] {result['image_name']}\n")
            f.write("-"*70 + "\n")
            
            # EasyOCR Stats
            f.write("\n📊 EasyOCR:\n")
            f.write(f"   Time: {result['easyocr']['time']:.2f}s\n")
            f.write(f"   Confidence: {result['easyocr']['confidence']:.2%}\n")
            f.write(f"   Words: {result['easyocr']['stats']['total_words']}\n")
            f.write(f"   Arabic: {result['easyocr']['stats']['arabic_percent']}%\n")
            f.write(f"   English: {result['easyocr']['stats']['english_percent']}%\n")
            f.write(f"   Preview: {result['easyocr']['text'][:200]}...\n")
            
            # Tesseract Stats
            if result['tesseract']['success']:
                f.write("\n📊 Tesseract:\n")
                f.write(f"   Time: {result['tesseract']['time']:.2f}s\n")
                f.write(f"   Words: {result['tesseract']['stats']['total_words']}\n")
                f.write(f"   Arabic: {result['tesseract']['stats']['arabic_percent']}%\n")
                f.write(f"   English: {result['tesseract']['stats']['english_percent']}%\n")
                f.write(f"   Preview: {result['tesseract']['text'][:200]}...\n")
            
            # Comparison
            f.write("\n⚖️ Comparison:\n")
            if result['tesseract']['success']:
                word_diff = result['easyocr']['stats']['total_words'] - result['tesseract']['stats']['total_words']
                f.write(f"   Word difference: EasyOCR {word_diff:+d} vs Tesseract\n")
            f.write(f"   Speed: EasyOCR {result['easyocr']['time']:.2f}s vs Tesseract {result['tesseract']['time']:.2f}s\n")
            
            if result['easyocr']['time'] < result['tesseract']['time']:
                speed_advantage = (result['tesseract']['time'] / result['easyocr']['time'] - 1) * 100
                f.write(f"   ✓ EasyOCR is {speed_advantage:.1f}% faster\n")
            else:
                speed_advantage = (result['easyocr']['time'] / result['tesseract']['time'] - 1) * 100
                f.write(f"   ✓ Tesseract is {speed_advantage:.1f}% faster\n")
            
            f.write("\n" + "-"*70 + "\n")
        
        # Summary
        f.write("\n" + "="*70 + "\n")
        f.write("SUMMARY\n")
        f.write("="*70 + "\n")
        f.write(f"Total images processed: {len(results)}\n")
        f.write(f"Languages: Arabic + English (Bilingual)\n")
        f.write(f"EasyOCR: Better accuracy, slower\n")
        f.write(f"Tesseract: Faster, needs preprocessing\n")
    
    print(f"\n✓ {report_file.name}")

# ============================================================
# STEP 13: DISPLAY RESULTS IN CONSOLE
# ============================================================

def print_detailed_results(results):
    """
    Print formatted results to console
    """
    print("\n" + "="*70)
    print("📄 DETAILED RESULTS")
    print("="*70)
    
    for idx, result in enumerate(results, 1):
        print(f"\n[IMAGE {idx}] {result['image_name']}")
        print("-"*70)
        
        print("\n🟦 EASYOCR (Arabic + English):")
        print(f"  Words: {result['easyocr']['stats']['total_words']}")
        print(f"  Characters: {result['easyocr']['stats']['total_chars']}")
        print(f"  Arabic: {result['easyocr']['stats']['arabic_percent']}%")
        print(f"  English: {result['easyocr']['stats']['english_percent']}%")
        print(f"  Time: {result['easyocr']['time']:.2f}s")
        print(f"\n  Text preview (first 300 chars):")
        print(f"  {result['easyocr']['text'][:300]}")
        
        if result['tesseract']['success']:
            print("\n🟨 TESSERACT (Arabic + English):")
            print(f"  Words: {result['tesseract']['stats']['total_words']}")
            print(f"  Characters: {result['tesseract']['stats']['total_chars']}")
            print(f"  Arabic: {result['tesseract']['stats']['arabic_percent']}%")
            print(f"  English: {result['tesseract']['stats']['english_percent']}%")
            print(f"  Time: {result['tesseract']['time']:.2f}s")
            print(f"\n  Text preview (first 300 chars):")
            print(f"  {result['tesseract']['text'][:300]}")

# ============================================================
# STEP 14: MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    
    # Run batch processing
    all_results = process_all_test_images()
    
    # Display results
    print_detailed_results(all_results)
    
    # Save everything
    save_results(all_results, OUTPUT_DIR)
    
    print("\n" + "="*70)
    print("✓ COMPLETE!")
    print("="*70)
    print(f"\nResults saved in: {OUTPUT_DIR}/")
    print("\nFiles created:")
    print("  - IMG_XXXX_EasyOCR.txt (extracted text from EasyOCR)")
    print("  - IMG_XXXX_Tesseract.txt (extracted text from Tesseract)")
    print("  - IMG_XXXX_metadata.json (detailed statistics)")
    print("  - COMPARISON_REPORT.txt (full analysis)")

Using CPU. Note: This module is much faster with a GPU.


✓ All libraries imported successfully!
✓ Directories configured:
  Training data: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\training
  Testing data: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing
  Results: C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\ocr_results

✓ Test images configured: 4 images
   - C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6208.jpg
   - C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6209.jpg
   - C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6212.jpg
   - C:\Users\Dell\Desktop\APZIVA\MonReader Project\OCR\testing\IMG_6213.jpg

INITIALIZING OCR ENGINES
🔄 Loading EasyOCR (Arabic + English)...
✓ EasyOCR Ready
✓ Tesseract Ready

🚀 BATCH PROCESSING TEST IMAGES

[1/4] 
📍 Processing: IMG_6208.jpg
----------------------------------------------------------------------
🔄 EasyOCR (Arabic + English)...
  ✓ Complete in 162.33s
  ✓ 242 words extracted
  ✓ Arabic: 79.3% | English: 0.0%
  ✓ Confidenc